# Universal Multi-Agent Communication & Orchestration Guide

Welcome to the **Universal Guide for the Modular Agent Runtime**! This notebook demonstrates the core capabilities of the refactored, strongly-typed Ravi Agent Framework. We will explore how to build modern, stable, long-running agent workflows using the pure Python actor runtime.

## Key Architectural Pillars Demonstrated:
1. **Universal Multimodal Contracts**: All message payloads are strictly typed as `list[ContentBlock]` (e.g., `TextBlock` packages), completely eliminating legacy/backward-compatible dictionary parsing and `Any` types.
2. **User-to-Agent Interaction**: High-level interface where the user publishes or sends messages directly to an actor agent.
3. **Agent-to-Agent P2P Messaging**: Dynamic communication where agents invoke, consult, or pass messages to peer agents asynchronously through `MessageContext`.
4. **Agent-to-Subagent Orchestration**: Decomposing complex monolithic agents into modular parent-child hierarchies using direct orchestration handoffs.

---

## 1. Environment & Runtime Bootstrap

First, let's import the core components: `LocalRuntime`, `AgentId`, `MessageContext`, and `TextBlock` (the standard multimodal message block).

In [1]:
import asyncio
from ravi.kernel.runtime import LocalRuntime, AgentId, MessageContext
from ravi.kernel.messages.content import TextBlock

print("✅ Core imports loaded successfully!")

✅ Core imports loaded successfully!


## 2. Scenario 1: User-to-Agent Communication

In this scenario, a human user sends a prompt directly to a registered agent type. The agent processes the message and returns a fully-typed result.

Let's register a simple **Translator Agent** that translates text into Spanish.

In [2]:
async def translator_handler(ctx: MessageContext, content: list[TextBlock]) -> str:
    """Translates any incoming text into Spanish."""
    text = content[0].text if content else ""
    print(f"[Translator] Received message from user: '{text}'")
    # Core business logic: translate
    translated = f"{text} -> ¡Hola! Traducido al español."
    return translated

async def run_user_to_agent_demo():
    # Initialize and start the LocalRuntime
    runtime = LocalRuntime()
    await runtime.start()
    
    # Register the translator agent type
    await runtime.register("translator", translator_handler)
    
    # Send message from the 'caller' to 'translator' instance
    recipient = AgentId(type="translator", key="instance-1")
    result = await runtime.send_message(
        message="Hello, how are you?",
        sender=AgentId(type="caller", key="user-session"),
        recipient=recipient,
    )
    
    print(f"\n[User] Result from translator: '{result}'")
    await runtime.stop()

# Run the scenario
await run_user_to_agent_demo()

[Translator] Received message from user: 'Hello, how are you?'

[User] Result from translator: 'Hello, how are you? -> ¡Hola! Traducido al español.'


## 3. Scenario 2: Agent-to-Agent Communication (P2P Mesh)

Agents in a production environment rarely work in isolation. Often, one agent needs to consult another agent. 

Here, we build a **Supervisor Agent** that receives a mathematical expression, delegates the actual execution to a **Calculator Agent**, and wraps the final output for the user.

In [3]:
async def calculator_handler(ctx: MessageContext, content: list[TextBlock]) -> str:
    """Calculates simple mathematical evaluations securely."""
    expression = content[0].text if content else ""
    print(f"  [Calculator] Computing: {expression}")
    try:
        # Secure eval of simple numbers
        result = str(eval(expression, {"__builtins__": None}))
    except Exception as e:
        result = f"Error: {str(e)}"
    return result

async def supervisor_handler(ctx: MessageContext, content: list[TextBlock]) -> str:
    """Supervisor receives request, asks Calculator, and returns formatted reply."""
    task = content[0].text if content else ""
    print(f"[Supervisor] Received task: '{task}'")
    
    # Agent-to-Agent P2P messaging using the runtime context reference!
    calc_recipient = AgentId(type="calculator", key="calc-service")
    calc_result = await ctx.runtime.send_message(
        message=task,
        sender=ctx.agent_id,
        recipient=calc_recipient
    )
    
    return f"Supervisor Report: The result for '{task}' is {calc_result}."

async def run_agent_to_agent_demo():
    runtime = LocalRuntime()
    await runtime.start()
    
    # Register both agents
    await runtime.register("calculator", calculator_handler)
    await runtime.register("supervisor", supervisor_handler)
    
    # Send message from user to supervisor
    result = await runtime.send_message(
        message="45 * 2 + 10",
        recipient=AgentId(type="supervisor", key="main-supervisor"),
    )
    
    print(f"\n[User] Result from supervisor: '{result}'")
    await runtime.stop()

# Run the scenario
await run_agent_to_agent_demo()

[Supervisor] Received task: '45 * 2 + 10'
  [Calculator] Computing: 45 * 2 + 10

[User] Result from supervisor: 'Supervisor Report: The result for '45 * 2 + 10' is 100.'


## 4. Scenario 3: Agent-to-Subagent Orchestration

Now let's demonstrate direct **parent-child subagent delegation** using the refactored runtime orchestration patterns.

We have an **Orchestrator Agent** (parent) that delegates specialized questions to focused subagents:
- **Billing Specialist** (child) handles invoice/billing queries.
- **Technical Support Specialist** (child) handles system error queries.

The orchestrator uses typed handoff dispatching to execute the subagent through the runtime, providing flawless message flow and isolation.

In [4]:
async def billing_handler(ctx: MessageContext, content: list[TextBlock]) -> str:
    query = content[0].text if content else ""
    print(f"    [Billing Specialist] Answering query: '{query}'")
    return f"Billing Response: Checked records. Invoice for '{query}' has been fully paid."

async def tech_support_handler(ctx: MessageContext, content: list[TextBlock]) -> str:
    query = content[0].text if content else ""
    print(f"    [Tech Support] Diagnosing error: '{query}'")
    return f"Tech Support Response: Internal database lock found on '{query}'. Auto-resolved successfully."

async def orchestrator_handler(ctx: MessageContext, content: list[TextBlock]) -> str:
    query = content[0].text if content else ""
    print(f"[Orchestrator] Classifying query: '{query}'")
    
    # Parent routes message dynamically to subagents based on simple keywords
    if "invoice" in query.lower() or "pay" in query.lower():
        subagent_recipient = AgentId(type="billing_specialist", key="billing-actor")
        print(f"[Orchestrator] routing query to Billing Specialist...")
    else:
        subagent_recipient = AgentId(type="tech_support", key="tech-actor")
        print(f"[Orchestrator] routing query to Tech Support...")
        
    # Delegate through the runtime framework seamlessly
    subagent_result = await ctx.runtime.send_message(
        message=query,
        sender=ctx.agent_id,
        recipient=subagent_recipient
    )
    
    return f"[Orchestrator Unified Response] -> {subagent_result}"

async def run_orchestrator_demo():
    runtime = LocalRuntime()
    await runtime.start()
    
    # Register parent and subagents
    await runtime.register("billing_specialist", billing_handler)
    await runtime.register("tech_support", tech_support_handler)
    await runtime.register("orchestrator", orchestrator_handler)
    
    # Query 1: Billing related
    print("\n--- Query 1: Billing Handoff ---")
    res1 = await runtime.send_message(
        message="Status of invoice INV-2026?",
        recipient=AgentId(type="orchestrator", key="parent-orchestrator"),
    )
    print(f"Result 1: {res1}")
    
    # Query 2: Tech support related
    print("\n--- Query 2: Tech Support Handoff ---")
    res2 = await runtime.send_message(
        message="Connection Timeout on Redis db 2",
        recipient=AgentId(type="orchestrator", key="parent-orchestrator"),
    )
    print(f"Result 2: {res2}")
    
    await runtime.stop()

# Run the scenario
await run_orchestrator_demo()


--- Query 1: Billing Handoff ---
[Orchestrator] Classifying query: 'Status of invoice INV-2026?'
[Orchestrator] routing query to Billing Specialist...
    [Billing Specialist] Answering query: 'Status of invoice INV-2026?'
Result 1: [Orchestrator Unified Response] -> Billing Response: Checked records. Invoice for 'Status of invoice INV-2026?' has been fully paid.

--- Query 2: Tech Support Handoff ---
[Orchestrator] Classifying query: 'Connection Timeout on Redis db 2'
[Orchestrator] routing query to Tech Support...
    [Tech Support] Diagnosing error: 'Connection Timeout on Redis db 2'
Result 2: [Orchestrator Unified Response] -> Tech Support Response: Internal database lock found on 'Connection Timeout on Redis db 2'. Auto-resolved successfully.


## Summary

This robust modular runtime eliminates complex dictionary logic in favor of strictly typed universal contracts (`list[ContentBlock]` and `TextBlock`). You've seen user-to-agent, P2P agent-to-agent, and nested parent-child subagent orchestration workflows executing with maximum type-safety, resilience, and clarity! 🎉